# Water Potability Prediction

**Beginner Data Analytics + Machine Learning Project**

### Objective
Analyze water-quality measurements and build classification models to predict the `Potability` label.

> `Potability = 1` means the dataset labels the sample as potable; `0` means non-potable.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# 2. Load the dataset
df = pd.read_csv("../data/water_potability.csv")

print("Dataset shape:", df.shape)
df.head()

## 3. Understand the data

First check column names, data types, missing values and duplicate rows.

In [ ]:
# Basic information
df.info()

In [ ]:
# Summary statistics
df.describe().T

In [ ]:
# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing

In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

## 4. Explore the target variable

In [ ]:
# Potability distribution
print(df["Potability"].value_counts())

sns.countplot(data=df, x="Potability")
plt.title("Water Potability Distribution")
plt.xlabel("Potability (0 = Non-Potable, 1 = Potable)")
plt.ylabel("Number of Samples")
plt.show()

## 5. Explore feature distributions

In [ ]:
# Histograms for numerical variables
df.hist(figsize=(14, 10), bins=25)
plt.suptitle("Distribution of Water Quality Features")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 6. Handle missing values

Instead of dropping rows, use median imputation inside a Scikit-learn pipeline. This keeps preprocessing reproducible and prevents test-set information from leaking into training.

In [ ]:
X = df.drop(columns="Potability")
y = df["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Train Logistic Regression

Logistic Regression is a simple baseline classification model.

In [ ]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)

print("Logistic Regression Accuracy:",
      round(accuracy_score(y_test, logistic_pred), 4))
print()
print(classification_report(y_test, logistic_pred))

## 8. Train Random Forest

Random Forest is a tree-based classification model and gives us a second model for comparison.

In [ ]:
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:",
      round(accuracy_score(y_test, rf_pred), 4))
print()
print(classification_report(y_test, rf_pred))

## 9. Compare model performance

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, rf_pred)
    ]
})

results.sort_values("Accuracy", ascending=False)

## 10. Confusion Matrix

The confusion matrix shows correct and incorrect predictions for each class.

In [ ]:
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Non-Potable", "Potable"],
    yticklabels=["Non-Potable", "Potable"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

## 11. Feature importance

Random Forest can provide an estimate of which features contributed most to its predictions.

In [ ]:
rf = rf_model.named_steps["model"]

importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
importance.sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.show()

importance

# Final Takeaways

1. The dataset contains multiple water-quality measurements and a binary potability label.
2. Missing values should be handled before model training.
3. We compared a simple Logistic Regression baseline with Random Forest.
4. Accuracy alone should not be treated as the only measure of model quality, especially when the target classes are not perfectly balanced.
5. This model is an educational prediction exercise, not a substitute for laboratory water-quality testing.